In [93]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F

In [94]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [95]:
class RNN1(nn.Module):
    def __init__(self, input_size, hidden_size, hidden_sleep_size, context_size, output_size=7, num_layers=1, modulation_strength=0):
        super(RNN1, self).__init__()

        self.context_fc = nn.Linear(hidden_sleep_size, context_size, bias=False)
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.modulation_strength = modulation_strength

    def set_modulation(self, val):
        self.modulation_strength = val
        
    def forward(self, x, hs, h=None):
        context = self.context_fc(hs)
        # print(hs, context, 'sjs')
        x = torch.cat(((1-self.modulation_strength)*x, self.modulation_strength*context), dim=2)
        # print(x)
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h

In [96]:
class RNN2(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNN2, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, h=None):
        out, h = self.rnn(x, h)
        out = self.fc(out[:,-1,:])

        return out, h

In [97]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [98]:
### initial training ###
total_samples = 20000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 100
context_output_size = 10
num_layers_wake = 1
num_layers_sleep = 1
output_size = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-3
test_acc = []
c = []
cortical_strength = .5

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = RNN1(input_size+context_output_size, hidden_wake_size, hidden_sleep_size, context_size=context_output_size, output_size=output_size, num_layers=num_layers_wake)
network1.set_modulation(cortical_strength)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
hs = torch.zeros((1,1,hidden_sleep_size))
for X, y in train_loader:
    # context = torch.zeros((1,X.size(1),context_output_size))
    # X = torch.cat((X,context), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, h = network1(X,hs)
    else:
        predicted_y, h = network1(X, hs, h=mem)
    
    loss = criterion(predicted_y, y)
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = h.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 2.0133, accuracy: 0.2420
Iter : 2001, loss: 1.7415, accuracy: 0.2660
Iter : 3001, loss: 1.7029, accuracy: 0.4110
Iter : 4001, loss: 2.8130, accuracy: 0.5010
Iter : 5001, loss: 0.9385, accuracy: 0.4980
Iter : 6001, loss: 2.2957, accuracy: 0.6030
Iter : 7001, loss: 1.3856, accuracy: 0.6220
Iter : 8001, loss: 2.2115, accuracy: 0.6210
Iter : 9001, loss: 2.5449, accuracy: 0.6590
Iter : 10001, loss: 2.0235, accuracy: 0.6520
Iter : 11001, loss: 2.0603, accuracy: 0.6430
Iter : 12001, loss: 1.7704, accuracy: 0.6570
Iter : 13001, loss: 2.1172, accuracy: 0.6640
Iter : 14001, loss: 3.1840, accuracy: 0.6590
Iter : 15001, loss: 1.7790, accuracy: 0.6690
Iter : 16001, loss: 2.0645, accuracy: 0.6630
Iter : 17001, loss: 1.7144, accuracy: 0.6570
Iter : 18001, loss: 2.0212, accuracy: 0.6650
Iter : 19001, loss: 2.2059, accuracy: 0.6650


In [131]:
total_samples = 1000
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
cortical_strength = .01
SWR = .8
counts = []
seq = ''

network1.set_modulation(cortical_strength)
hs = torch.randn((1,1,hidden_sleep_size))
network2 = RNN2(hidden_wake_size, hidden_sleep_size, hidden_wake_size, num_layers=num_layers_sleep)
optimizer = torch.optim.SGD(network2.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.MSELoss()

prev_h = torch.randn((1,1,hidden_wake_size))
mem = hs.clone()
for jj in range(total_samples):
    with torch.no_grad():
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs)
        else:
            hidden_state += SWR*torch.randn(hidden_state.size())
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hs, hidden_state)

        h_target = hidden_state.clone()

    ### train RNN2 ###
    optimizer.zero_grad()
    predicted_h, hs = network2(prev_h, mem)
    loss = criterion(predicted_h[0][0], h_target[0][0])
    
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem = hs.clone()
        X_hat_prob = torch.nn.functional.softmax(X_hat_, dim=1)
        dist_categ = torch.distributions.Categorical(probs=X_hat_prob.reshape(-1))
        idx = dist_categ.sample()
        # idx = X_hat_prob.argmax()

        # noise = 10*torch.randn(context_output_size)
        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        # X_hat[len(tokens):] = noise
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]
        prev_h = hidden_state.clone()

        if jj%10000==0:
            print(loss)

        

tensor(0.2888, grad_fn=<MseLossBackward0>)


In [132]:
seq[-200:]

'GGACAGEDFEGBCGGACBGGBCCGCBGGCBGACBGCAGBCGFDGEFGEDGGEFGGBAGGCGDGEFDGFGGEDGGEFDDGCBGACBGGDGBAGGCBBCGFDGBAACGDFDEGFEFDGFGACBGACGGCAGEBGBCCGFDEDGBCGGCBGCBACGFDFGFEDGCBAGBACCGCACGACGGBCAGDCAGEDAGCGABGCGDFG'

In [332]:
hs

tensor([[[0.1832, 0.1243, 0.0832, 0.1566, 0.0000, 0.0932, 0.1200, 0.0121,
          0.0000, 0.2793, 0.2477, 0.0110, 0.0192, 0.0000, 0.0000, 0.0000,
          0.1732, 0.0868, 0.0802, 0.0000, 0.0000, 0.0000, 0.1326, 0.0000,
          0.0000, 0.0921, 0.2426, 0.0771, 0.0505, 0.0718, 0.0000, 0.2149,
          0.0000, 0.0000, 0.0022, 0.0372, 0.0330, 0.0000, 0.0445, 0.0000,
          0.0110, 0.0000, 0.0000, 0.0000, 0.0000, 0.0314, 0.1639, 0.0456,
          0.0000, 0.0000, 0.0626, 0.0502, 0.0699, 0.1655, 0.0000, 0.0967,
          0.0876, 0.0000, 0.1185, 0.0000, 0.0000, 0.1434, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2022, 0.1125, 0.0000,
          0.0289, 0.0000, 0.0055, 0.0000, 0.0000, 0.0000, 0.0387, 0.3551,
          0.0646, 0.0790, 0.0000, 0.0845, 0.0000, 0.2341, 0.0876, 0.0366,
          0.1313, 0.0000, 0.0000, 0.0000, 0.0000, 0.1756, 0.0630, 0.0000,
          0.0000, 0.0000, 0.0291, 0.0000]]], grad_fn=<StackBackward0>)

In [333]:
hidden_state[0][0]

tensor([0.0542, 0.0000, 0.0000, 0.0613, 0.0000, 0.4199, 0.3931, 0.0335, 0.2994,
        0.3308, 0.1982, 0.0000, 0.0000, 0.0000, 0.0000, 0.0223, 0.4450, 0.1165,
        0.3714, 0.0000, 0.0000, 0.0000, 0.2346, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0490, 0.8606, 0.3041, 0.4732, 0.0000, 0.0000, 0.4482, 0.5053, 0.3618,
        0.0000, 0.0000, 0.0000, 0.4408])